# Figure: cross-region coupling with xgboost

**Which regions predict which?** Every target cell is modelled twice: a
**baseline** using position (sin/cos — the track is a ring), running speed and
the cell's own spike history, and a **full** model that adds the binned spiking
of a 10-cell partner pool. Coupling is the gain

$$\Delta pR^2 = pR^2(\text{full}) - pR^2(\text{base})$$

so it measures what ten other cells add *on top of* everything the animal's own
behaviour already explains — not how well the pool predicts spiking in
absolute terms.

The heavy part (3 workers, ~11 h) is `cross_region/xr_full.py`, run separately;
this notebook reads its shards and builds the figure. The whole chain lives in
the repo so the shards can be regenerated:

    cross_region/xr_phase0.py    loaders, probe geometry, the xgboost model
    cross_region/xr_pc1.py       each cell's PC1-following score
    cross_region/xr_pilot.py     folds, cross-state pR², the pilot gates
    cross_region/xr_full.py      the full run  ->  data/cross_region_xgboost/

### Design

| | |
|---|---|
| **pool size** | fixed at 10 cells everywhere, so no cell of the matrix is advantaged by having more partners available |
| **diagonal** | same-region pool, but only cells ≥100 µm away on the same shank or on a different shank — keeps spike-sorting bleed between neighbouring contacts out of the ceiling |
| **targets** | one cell per decile of PC1-following, so each region's rank axis is covered by construction rather than by luck |
| **draws** | two random pools, averaged |
| **floor** | the `shift` condition re-runs each cross-region pool with every partner's spike train circularly rolled: same cells, same rates, same history structure, timing destroyed |

Every matrix value is floor-subtracted. The floor turns out to be ±0.003
everywhere, so subtraction barely moves anything — but it is what licenses
reading the matrix as *coupling* rather than as a covariate-count effect, and
it is **not** optional in the state analysis, where a baseline that generalises
badly across states makes any added covariate look helpful.

## The matrix cannot be read across rows

Each cell rests on a different set of sessions — 27 have enough MEC cells, 16
VIS, 7 SUB, 1 LEC — and sessions differ enormously in overall coupling. The
same quantity, MEC←MEC, is **+0.067** among the 12 sessions that also carry
subiculum and **+0.022** among the 21 that carry visual cortex: three times
apart, purely from which sessions contribute.

So panel A is the descriptive summary and **panel C carries the claims**, with
every comparison paired within session.

In [1]:
"""Cross-region coupling figure.

Run from the script rather than copied into the notebook, so the notebook and
the paper figure cannot drift apart. `_g` keeps the script's namespace, so the
tables it builds stay available below:

    _g['D']    every row of the run (cell x partner x condition x draw)
    _g['G']    one floor-subtracted dpR2 per session x target x partner
    _g['MAT']  the 4x4 matrix itself
    _g['ST']   the per-state frame behind panel E
"""
import runpy
from IPython.display import display

FIGDIR = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
          'AnchorDynamics2026')
_g = runpy.run_path(f'{FIGDIR}/cross_region_coupling.py')
display(_g['fig'])

3942 rows from 3 shards, 27 sessions

matrix (rows predicted <- columns predictor), floor-subtracted dpR2:
partner           MEC     SUB     VIS     LEC
target_region                                
MEC            0.0424  0.0672  0.0127  0.0104
SUB            0.1068  0.1367  0.0498     NaN
VIS            0.0100  0.0274  0.0126  0.0032
LEC            0.0068     NaN  0.0049  0.0162

sessions per cell:
partner        MEC  SUB  VIS  LEC
target_region                    
MEC             27   12   21    2
SUB              7    7    4    0
VIS             16    3   16    2
LEC              1    0    1    1

controls, pooled over sessions:
  within-region pool       +0.0444  (n = 51 session x pair)
  cross-region pool        +0.0307  (n = 74 session x pair)
  time-shifted (floor)     -0.0010  (n = 74 session x pair)

within-session paired tests:
  MEC<-MEC +0.0670 vs MEC<-SUB +0.0672  n=12  p=1.000
  MEC<-MEC +0.0223 vs MEC<-VIS +0.0127  n=21  p=0.000
  MEC<-MEC vs MEC<-CB: only 3 sessions, sk


saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig5_cross_region_coupling.pdf
saved coupling_matrix.csv and coupling_per_session.csv


<Figure size 740x920 with 6 Axes>

## What it shows

**Subicular pools predict MEC cells as well as other MEC cells do** — +0.067 vs
+0.067, n = 12 sessions, p = 1.0 — while visual pools predict them clearly
worse, +0.013 vs +0.022, n = 21, p < 0.001. Coupling is not detectably
asymmetric in either direction that has enough sessions to test (MEC↔VIS
p = 0.40, MEC↔SUB p = 0.22).

**Coupling does not track the anchoring state.** ΔpR² in anchored and
non-anchored trials is identical to three decimals in both regions with enough
sessions (MEC +0.0275 vs +0.0273, n = 18, p = 0.64; VIS +0.0104 vs +0.0106,
n = 13, p = 0.89). This is a real negative result for the idea that anchoring
is a change in how strongly cells are coupled to one another — whatever the
anchoring state is, it is not visible as pairwise predictability.

### Caveats the panels carry themselves

- **LEC is one session.** It is in the matrix because the matrix is 4×4; its
  row and column are not a test.
- **Cerebellum has no row.** No session has enough cerebellar cells to supply
  targets, so CB appears only as a partner pool (3 sessions against MEC
  targets, too few for the paired test). It is the distant negative control.
- **ΔpR² rises with a target's spike count** (rho = +0.20, p = 3e-24), which is
  why every comparison is paired on the same target cells wherever possible.
- **The diagonal has no shift condition of its own.** It is floor-subtracted
  using the session's mean cross-region floor, which assumes the floor does not
  depend on which region the (timing-destroyed) partners came from.

In [2]:
"""The matrix as numbers, and where each cell's sessions come from."""
MAT, NMAT, G = _g['MAT'], _g['NMAT'], _g['G']
print('floor-subtracted dpR2 (rows predicted <- columns predictor):')
print(MAT.round(4).to_string())
print('\nsessions contributing to each cell:')
print(NMAT.fillna(0).astype(int).to_string())
print('\nper-session spread, to show why rows are not comparable:')
print(G.groupby(['target_region', 'partner']).dc
      .agg(['mean', 'std', 'min', 'max', 'size']).round(4).to_string())

floor-subtracted dpR2 (rows predicted <- columns predictor):
partner           MEC     SUB     VIS     LEC
target_region                                
MEC            0.0424  0.0672  0.0127  0.0104
SUB            0.1068  0.1367  0.0498     NaN
VIS            0.0100  0.0274  0.0126  0.0032
LEC            0.0068     NaN  0.0049  0.0162

sessions contributing to each cell:
partner        MEC  SUB  VIS  LEC
target_region                    
MEC             27   12   21    2
SUB              7    7    4    0
VIS             16    3   16    2
LEC              1    0    1    1

per-session spread, to show why rows are not comparable:
                         mean     std     min     max  size
target_region partner                                      
LEC           LEC      0.0162     NaN  0.0162  0.0162     1
              MEC      0.0068     NaN  0.0068  0.0068     1
              VIS      0.0049     NaN  0.0049  0.0049     1
MEC           CB       0.0071  0.0014  0.0057  0.0085     3
    

In [3]:
"""Sanity check: the floor really is zero, condition by condition.

If this ever stops being true the matrix stops meaning coupling, because a
pool of ten covariates whose timing has been destroyed would be improving the
fit on structure alone.
"""
import numpy as np
D = _g['D']
f = (D[D.cond == 'shift']
     .groupby(['mouse', 'day', 'target_region', 'partner']).d_all.mean())
print(f'shifted floor: {f.mean():+.5f} +- {f.std(ddof=1) / np.sqrt(len(f)):.5f} '
      f'(n = {len(f)} session x pair)')
print(f'  range {f.min():+.4f} to {f.max():+.4f}')
print(f'  fraction above +0.01: {(f > .01).mean():.3f}')

shifted floor: -0.00104 +- 0.00018 (n = 74 session x pair)
  range -0.0054 to +0.0032
  fraction above +0.01: 0.000
